# Day 4 — Model Evaluation & Regularization

**Agri-AI Bootcamp** · Phase 1 of 5: *ML Fundamentals* · Days 1–4

> Add manual L2 to the Day 2 model and use GridSearchCV to find the best α on noisy weather data.

**Runtime:** a free CPU runtime is enough for today.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Overfitting means memorising noise; underfitting means missing the pattern. L2 shrinks all weights smoothly, L1 can zero some out. F1 combines precision and recall; GridSearchCV tries candidate α values with cross-validation.

### 🌾 Agriculture example

One odd drought year drags an unregularised yield model around; Ridge with α chosen by GridSearchCV gives steadier predictions on the next season.

> **Watch out:** Tune on validation folds, never on the final test set.

## 2. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

## 3. Data

In [ ]:
# ---- Farm-season data --------------------------------------------------------
# Course dataset: Kaggle "Crop Yield Prediction".  Kaggle needs an API token, so
# we simulate 200 farm-seasons with the same columns.  Replace this cell with
# pd.read_csv("yield_df.csv") once you have downloaded the real file.
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 200
year      = rng.integers(2005, 2025, n)
rainfall  = rng.normal(850, 180, n).clip(300, 1500)      # mm / season
fertiliser= rng.normal(120,  35, n).clip(20, 260)        # kg N / ha
# "true" relationship + noise, plus one drought year that behaves differently
yield_t   = (0.9 + 0.0022 * rainfall + 0.020 * fertiliser
             - 0.0000012 * rainfall ** 2 + rng.normal(0, 0.45, n))
yield_t[year == 2012] -= 1.8                             # drought season
df = pd.DataFrame(dict(year=year, rainfall=rainfall,
                       fertiliser=fertiliser, yield_t_ha=yield_t.clip(0.2, None)))
X = df[["rainfall", "fertiliser", "year"]].to_numpy(float)
y = df["yield_t_ha"].to_numpy(float)
df.head()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0)
print("train", Xtr.shape, "test", Xte.shape)

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
gs=GridSearchCV(Ridge(),{"alpha":[0.01,0.1,1,10,100]},cv=5).fit(Xtr,ytr)
print(gs.best_params_)
```

## 5. Worked solution

In [ ]:
# ---- Make overfitting visible: high-degree polynomial on few features --------
def rmse(model, A, b): return np.sqrt(mean_squared_error(b, model.predict(A)))

for deg in [1, 3, 6]:
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(deg), LinearRegression())
    pipe.fit(Xtr, ytr)
    print(f"degree {deg}: train RMSE {rmse(pipe,Xtr,ytr):.3f} | "
          f"test RMSE {rmse(pipe,Xte,yte):.3f}   <- gap = overfitting")

# ---- L2 (Ridge) with alpha chosen by cross-validation ------------------------
grid = GridSearchCV(make_pipeline(StandardScaler(), PolynomialFeatures(3), Ridge()),
                    {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                    cv=5, scoring="neg_root_mean_squared_error").fit(Xtr, ytr)
print(f"\nbest alpha {grid.best_params_['ridge__alpha']} "
      f"| CV RMSE {-grid.best_score_:.3f} | test RMSE {rmse(grid,Xte,yte):.3f}")
print("NOTE: alpha was chosen on validation folds, never on the test set.")

# ---- L1 (Lasso) zeroes features out -----------------------------------------
las = make_pipeline(StandardScaler(), Lasso(alpha=0.1)).fit(Xtr, ytr)
coef = las[-1].coef_
print("\nLasso coefficients:", dict(zip(["rainfall","fertiliser","year"], coef.round(3))),
      "\nzeroed:", [n for n, c in zip(["rainfall","fertiliser","year"], coef) if c == 0])

# ---- manual L2 in the Day-2 gradient loop (same idea, no library) ------------
mu, sd = Xtr.mean(0), Xtr.std(0)
Z, Zt = (Xtr - mu) / sd, (Xte - mu) / sd

def fit_l2(Z, y, lam=0.0, lr=0.05, epochs=1500):
    w = np.zeros(Z.shape[1]); b = 0.0
    for _ in range(epochs):
        err = Z @ w + b - y
        w -= lr * (2 * (Z.T @ err) / len(y) + 2 * lam * w)   # penalty term
        b -= lr * 2 * err.mean()
    return w, b

for lam in [0.0, 0.1, 1.0]:
    w, b = fit_l2(Z, ytr, lam)
    te = np.sqrt(((Zt @ w + b - yte) ** 2).mean())
    print(f"lambda {lam:<4} |w| = {np.abs(w).sum():.3f}  test RMSE {te:.3f}")

# ---- alpha vs error curve ----------------------------------------------------
al = np.logspace(-2, 3, 12)
cv = [-cross_val_score(make_pipeline(StandardScaler(), PolynomialFeatures(3), Ridge(alpha=a)),
                       Xtr, ytr, cv=5, scoring="neg_root_mean_squared_error").mean() for a in al]
plt.semilogx(al, cv, marker="o"); plt.xlabel("alpha"); plt.ylabel("CV RMSE")
plt.title("regularisation strength vs validation error"); plt.show()

## 6. Your turn

1. Run the same GridSearchCV with the 2012 drought rows removed. Does the best alpha drop?
2. Swap RMSE for R2 in the scoring and check the best alpha does not change much.
3. For a classifier, F1 replaces RMSE — write one line saying when you would use F1 over accuracy.

In [ ]:
# TODO 1: drop the drought season and re-tune
mask = df.year.to_numpy() != 2012
g2 = GridSearchCV(make_pipeline(StandardScaler(), PolynomialFeatures(3), Ridge()),
                  {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                  cv=5, scoring="neg_root_mean_squared_error").fit(X[mask], y[mask])
print("best alpha without 2012:", g2.best_params_)

# TODO 2: scoring="r2"
# TODO 3: one line on F1 vs accuracy -- when does accuracy lie?

## 7. Dataset and references

**Dataset:** [Kaggle: Crop Yield Prediction](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset)

**References**

- [Géron Hands-On ML notebooks](https://github.com/ageron/handson-ml3)
- [scikit-learn grid search](https://scikit-learn.org/stable/modules/grid_search.html)

**Course material:** Géron, Hands-On ML Ch.4

---
**Next:** Day 5 — Introduction to PyTorch & Tensors